# Background Term Domain Assignment — Jurist Review Tool

Questo notebook ha due fasi:
1. **FASE 1**: Genera il CSV di revisione con l'assegnamento k-NN di tutti gli 8.975 termini background
2. **FASE 2**: Dopo che hai annotato il CSV, carica i risultati e calcola le metriche di accuratezza

## Come annotare il CSV
Apri `lens_1_relational/results/background_review.csv` in Excel o Numbers.
Per ogni termine, compila la colonna `annotation`:
- **✓** → l'assegnamento è corretto
- **nome_dominio** → il dominio corretto (es. `criminal`, `civil`, `constitutional`, ...)
I termini sono ordinati per dominio e poi per confidence decrescente.
Parti dai termini ad alta confidence (confidence=1.0): saranno quasi tutti corretti.

## FASE 1 — Generazione del CSV

In [ ]:
import sys
from pathlib import Path
import numpy as np
from collections import Counter

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from shared.embeddings import load_precomputed
from lens_1_relational.domain_assignment import assign_domains, build_review_csv, load_annotations

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
RESULTS_DIR = ROOT / "lens_1_relational" / "results"
CSV_PATH = RESULTS_DIR / "background_review.csv"
MODEL = "BGE-EN-large"
K = 7

In [ ]:
print(f"Loading embeddings for {MODEL}...")
vecs, index = load_precomputed(MODEL, EMB_DIR)

core_idx  = [i for i, t in enumerate(index) if t["tier"] == "core" and t["domain"]]
bg_idx    = [i for i, t in enumerate(index) if t["tier"] == "background"]

terms_core = [index[i] for i in core_idx]
terms_bg   = [index[i] for i in bg_idx]
labels_core = [t["domain"] for t in terms_core]

vecs_core = vecs[core_idx]
vecs_bg   = vecs[bg_idx]

print(f"Core terms: {len(terms_core)} | Background terms: {len(terms_bg)}")
print(f"Running k-NN (k={K})...")

assignments = assign_domains(vecs_bg, vecs_core, labels_core, k=K)

print(f"Done. Domain distribution:")
counts = Counter(a["assigned_domain"] for a in assignments)
for d, n in sorted(counts.items()):
    print(f"  {d:25s}: {n:5d} ({100*n/len(assignments):.1f}%)")

In [ ]:
confidences = np.array([a["confidence"] for a in assignments])
low_conf_mask = confidences < 4/K

print("Confidence distribution:")
print(f"  1.00 (unanimità):  {(confidences == 1.0).sum():5d} ({100*(confidences==1.0).mean():.1f}%)")
print(f"  ≥0.57 (maggioranza solida): {(confidences >= 4/K).sum():5d} ({100*(confidences>=4/K).mean():.1f}%)")
print(f"  <0.57 (ambigui — revisione obbligatoria): {low_conf_mask.sum():5d} ({100*low_conf_mask.mean():.1f}%)")

print(f"\nGenerating CSV → {CSV_PATH}")
build_review_csv(terms_bg, assignments, terms_core, CSV_PATH)
print(f"✓ CSV saved with {len(terms_bg)} rows")
print("\nApri il file in Excel o Numbers per annotare la colonna 'annotation'.")

In [ ]:
print("=== PREVIEW: 2 esempi alta confidence per dominio ===\n")
from collections import defaultdict
by_domain = defaultdict(list)
for term, asgn in zip(terms_bg, assignments):
    by_domain[asgn["assigned_domain"]].append((term, asgn))

for domain in sorted(by_domain):
    entries = sorted(by_domain[domain], key=lambda x: -x[1]["confidence"])
    print(f"--- {domain.upper()} ({len(entries)} termini) ---")
    for term, asgn in entries[:2]:
        print(f"  [{asgn['confidence']:.2f}] {term['en']!r}")
        for j in range(3):
            nb = terms_core[asgn['neighbor_indices'][j]]
            print(f"    → {nb['en']!r:40s} [{asgn['neighbor_domains'][j]}] sim={asgn['neighbor_sims'][j]:.3f}")
    print()

## FASE 2 — Analisi delle annotazioni

Esegui questa sezione **dopo** aver annotato e salvato il CSV.

In [ ]:
if not CSV_PATH.exists():
    print("CSV non trovato. Esegui prima la Fase 1.")
else:
    results = load_annotations(CSV_PATH)

    if results["annotated"] == 0:
        print("Nessuna annotazione trovata nel CSV.")
        print("Compila la colonna 'annotation' e ri-esegui questa cella.")
    else:
        print(f"Annotati: {results['annotated']} / {results['total']} termini")
        print(f"Corretti: {results['correct']}")
        if results['accuracy_overall'] is not None:
            print(f"Accuracy: {results['accuracy_overall']:.1%}")

        print("\nAccuracy per dominio:")
        for domain, stats in sorted(results["per_domain"].items()):
            bar = "█" * int(stats["accuracy"] * 20)
            print(f"  {domain:25s} {stats['accuracy']:.1%}  {bar}  ({stats['correct']}/{stats['total']})")

        if results["errors"]:
            print(f"\nTermini mal-assegnati ({len(results['errors'])}):")
            for e in results["errors"][:20]:
                print(f"  {e['en']!r:40s}  assegnato={e['assigned']:15s} → corretto={e['corrected_to']}")
            if len(results["errors"]) > 20:
                print(f"  ... e altri {len(results['errors'])-20}")

## Note metodologiche

**Perché k=7?** k dispari evita i pareggi. Con 7 domini, confidence < 4/7 ≈ 0.57 indica ambiguità genuina.

**Cosa fare con i termini ambigui?** I termini con confidence bassa sono i "boundary objects" della tesi (§4.1): concetti che la geometria non riesce a classificare perché appartengono a più domini. La tua valutazione giuridica è il dato scientifico.

**Come riportare i risultati?** L'accuracy complessiva entra in §3.1.1. Le distorsioni per dominio entrano in §3.1.4. I casi di errore più interessanti entrano in Ch.4 (Quod Numeri Tacent).